# 01 · Exploring ESEF Inline XBRL reports

Before writing a parser, this notebook looks at what the 20 MVP reports actually contain:
how pages are marked up, whether tables are real `<table>` elements, and whether the tagged
facts in the XHTML can be matched to the xBRL-JSON published by filings.xbrl.org.

The raw files are not in the repository. Run `uv run ledgerlens ingest download` first to
reproduce the outputs.

In [1]:
import collections
import json
import re
from pathlib import Path

import pandas as pd
from lxml import etree

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
IX = "{http://www.xbrl.org/2013/inlineXBRL}"
XH = "{http://www.w3.org/1999/xhtml}"
manifest = pd.read_csv(ROOT / "data/manifest/filings_manifest.csv")
manifest[["company", "fiscal_year", "report_bytes"]].assign(
    report_mb=lambda d: (d.report_bytes / 1e6).round(1)
).drop(columns="report_bytes").pivot(index="company", columns="fiscal_year", values="report_mb")

fiscal_year,2022,2023,2024,2025
company,,,,
ASML,49.0,67.4,86.0,47.1
Adyen,44.3,90.3,141.1,84.8
Airbus,14.6,9.0,14.5,16.7
Novo Nordisk,30.5,36.7,46.6,37.2
Philips,10.7,18.5,30.6,65.4


Reports are large: 9 to 141 MB of XHTML. Most of that is base64-encoded images, but even the
text alone is 100k to 220k words per report (roughly 150k to 300k tokens), so a report never
fits comfortably in a prompt and retrieval is required.

## A single report: ASML FY2024

In [2]:
def load(lei: str, fy: int) -> etree._Element:
    path = ROOT / f"data/raw/filings/{lei}/{fy}/report.xhtml"
    return etree.parse(str(path), etree.XMLParser(huge_tree=True)).getroot()


ASML = "724500Y6DUVHQD6OXN27"
asml = load(ASML, 2024)
numeric = asml.findall(f".//{IX}nonFraction")
print(len(numeric), "numeric facts,", len(asml.findall(f".//{IX}nonNumeric")), "text facts")
pd.DataFrame([{**e.attrib, "displayed": "".join(e.itertext())} for e in numeric[6:12]])

590 numeric facts, 70 text facts


,unitRef,contextRef,decimals,name,format,scale,id,displayed
0,u-1,c-1,-5,ifrs-full:Revenue,ixt:num-dot-decimal,6,f-7,"21,173.4"
1,u-1,c-2,-5,ifrs-full:Revenue,ixt:num-dot-decimal,6,f-8,"27,558.5"
2,u-1,c-3,-5,ifrs-full:Revenue,ixt:num-dot-decimal,6,f-9,"28,262.9"
3,u-1,c-1,-5,asml:CostOfSystemSales,ixt:num-dot-decimal,6,f-10,"7,843.3"
4,u-1,c-2,-5,asml:CostOfSystemSales,ixt:num-dot-decimal,6,f-11,"10,439.1"
5,u-1,c-3,-5,asml:CostOfSystemSales,ixt:num-dot-decimal,6,f-12,"10,615.1"


Each `ix:nonFraction` carries a concept (`name`), a context (period and dimensions), a unit,
a `scale` (6 = millions), a `format` that says how to read the displayed text
(`num-dot-decimal` means `21,173.4`) and optionally `sign="-"`. The machine value is
`displayed × 10^scale`, negated if `sign` is set. These attributes are exactly what makes the
tags a precise answer key.

The displayed number lives inside an ordinary HTML table:

In [3]:
def clean(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()


fact = asml.xpath("//*[@id='f-9']")[0]
table = next(a for a in fact.iterancestors() if a.tag == XH + "table")
for tr in list(table.iter(XH + "tr"))[:6]:
    print(" | ".join(clean("".join(td.itertext())) for td in tr))

 |  |  |  |  | 
Year ended December 31 (€, in millions, except per share data) | Notes | 2022 | 2023 |  | 2024
Net system sales | 2 | 15,430.3 | 21,938.6 |  | 21,768.7
Net service and field option sales |  | 5,743.1 | 5,619.9 |  | 6,494.2
Total net sales | 2, 3 | 21,173.4 | 27,558.5 |  | 28,262.9
 |  |  |  |  | 


## Survey of all 20 reports

The ESEF regulation fixes the tagging, not the HTML. Each company uses a different converter,
so the structure varies a lot. For every report, count the page containers, real tables and
whether fact ids line up with the xBRL-JSON keys.

In [4]:
def page_family(root: etree._Element) -> str:
    body = root.find(XH + "body")
    classes = collections.Counter(
        c
        for e in body.iter(XH + "div")
        for c in (e.get("class") or "").split()
        if c in ("pageView", "page", "pf")
    )
    if classes:
        return max(classes, key=classes.get)
    kids = [k for k in body if "display:none" not in (k.get("style") or "").replace(" ", "")]
    return "body-children" if len(kids) > 50 else "flow (no pages)"


rows = []
for r in manifest.itertuples():
    root = load(r.lei, r.fiscal_year)
    facts = json.loads(
        (ROOT / f"data/raw/filings/{r.lei}/{r.fiscal_year}/report.json").read_text("utf-8")
    )["facts"]
    numeric = root.findall(f".//{IX}nonFraction")
    rows.append(
        {
            "company": r.company,
            "fy": r.fiscal_year,
            "pages": page_family(root),
            "tables": len(root.findall(f".//{XH}table")),
            "numeric_facts": len(numeric),
            "with_id": sum(1 for e in numeric if e.get("id")),
            "id_in_json": sum(1 for e in numeric if e.get("id") in facts),
        }
    )
survey = pd.DataFrame(rows)
survey

,company,fy,pages,tables,numeric_facts,with_id,id_in_json
0,Adyen,2022,pageView,293,293,293,293
1,Adyen,2023,pageView,368,299,299,299
2,Adyen,2024,pageView,470,318,318,318
3,Adyen,2025,pageView,702,321,321,321
4,Airbus,2022,flow (no pages),225,452,0,0
5,Airbus,2023,body-children,0,457,0,0
6,Airbus,2024,page,0,456,456,456
7,Airbus,2025,page,0,457,457,457
8,ASML,2022,pageView,1129,595,595,595
9,ASML,2023,pageView,1158,596,596,596


Three findings shape the parser:

1. **Page markup differs by converter.** `pageView` divs (ASML, Adyen, Philips 2024+),
   `page` divs (Airbus 2024+), pdf2htmlEX `pf` frames (Novo Nordisk), plain body children
   (Airbus 2023) or no pages at all (Airbus 2022, Philips 2022/23).
2. **Some reports have no `<table>` elements.** Airbus 2023+ and Novo Nordisk are PDF
   conversions where each text fragment is absolutely positioned with CSS.
3. **Fact ids are optional.** Several reports have no `id` on `ix:nonFraction`, so the parser
   must assign synthetic anchors and the cross-check must match facts by value, not by id.

## Rebuilding table rows from absolute positions (Airbus FY2024)

In [5]:
AIRBUS = "MINO79WLOO247M1IL051"
airbus = load(AIRBUS, 2024)
css = "".join(s.text or "" for s in airbus.findall(f".//{XH}style"))
rules = dict(re.findall(r"\.([\w-]+)\{([^}]*)\}", css))


def position(span):
    left = bottom = None
    for c in (span.get("class") or "").split():
        rule = rules.get(c, "")
        if m := re.search(r"left:\s*([\d.]+)px", rule):
            left = float(m.group(1))
        if m := re.search(r"bottom:\s*([\d.]+)px", rule):
            bottom = float(m.group(1))
    return left, bottom


fact = airbus.xpath("//*[@id='reg-ix-1']")[0]
page = next(a for a in fact.iterancestors() if "page" in (a.get("class") or "").split())
lines = collections.defaultdict(list)
for span in page.iter(XH + "span"):
    if "text" not in (span.get("class") or "").split():
        continue
    left, bottom = position(span)
    lines[round((bottom or 0) / 3)].append((left or 0, "".join(span.itertext()).strip()))
for key in sorted(lines, reverse=True)[6:16]:
    print(" | ".join(t for _, t in sorted(lines[key]) if t))

(In € million) | Note | 2024 | 2023
Revenue | 12 | 69,230 | 65,446
Cost of sales
(58,555) | (55,402)
Gross margin | 12
10,675 | 10,044
Selling expenses
(877) | (867)
Administrative expenses | (1,744) | (1,654)
Research and development expenses | 13 | (3,250) | (3,257)


Grouping fragments by their vertical position and sorting by horizontal position recovers
readable rows, but fixed 3-pixel buckets are not good enough: "Cost of sales" and its numbers
land in different buckets because the label and the figures sit on baselines one pixel apart
that straddle a bucket boundary. The parser therefore clusters fragments by distance to the
previous line instead of bucketing. The same idea works for pdf2htmlEX, where positions are
encoded in class names (`x19d`, `y16f9`).

## Implications for the design

* Parse pages and tables per converter family, falling back to positional line reconstruction.
* Keep the element id (or a synthetic anchor) of every tagged number inside each text block, so
  the evaluation can locate the evidence for a gold fact without exposing the tag to the agent.
* Resolve concept prefixes through the namespace map, never by string prefix.
* Revenue is tagged as `ifrs-full:Revenue` in most years but as
  `ifrs-full:RevenueFromContractsWithCustomers` in some (ASML FY2025, Airbus FY2022), so the
  answer key needs a small metric-to-concepts mapping.